In [3]:
import numpy as np
import h5py
from pathlib import Path

# ============================================================
# 1. Parameters
# ============================================================

root = Path(
    "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/local_analysis"
)

FF_path = root / (
    "FF_symmetric_q0_0_0_hyp0-1-2-3-4-5-6-7-8_fb_tins18_ncfg799.h5"
)

pt2_path = root / (
    "799cfgs/"
    "2pt_N40_rho3.25_G5_ez_momfrac0p6_fb_tsep18_ncfg799.h5"
)

# Array indices
TGF = 3
W = 1
PF = 5
TSEP = 2
TAU = 6

# "real" or "imag"
PART = "real"


# ============================================================
# 2. Jackknife utilities
# ============================================================

def jackknife(x):
    n = x.shape[0]
    return (np.sum(x, axis=0) - x) / (n - 1)


def jk_error(x):
    n = len(x)
    mean = np.mean(x)
    return np.sqrt(
        (n - 1) / n * np.sum(np.abs(x - mean)**2)
    )


# ============================================================
# 3. Load original lattice data (READ ONLY)
# ============================================================

with h5py.File(FF_path, "r") as f, \
     h5py.File(pt2_path, "r") as p:

    # Verify configuration alignment if both lists exist
    if "cfg_list" in f and "cfg_list" in p:
        assert np.array_equal(
            f["cfg_list"][:],
            p["cfg_list"][:]
        ), "Configuration lists are not aligned!"

    ff = f["FF_forward"]

    # O3 = TXTX + TYTY - 2 XYXY
    # Shape: [cfg, tsrc]
    G = (
        ff[:, :, 3, 3, TGF, W, 0, TAU]
        + ff[:, :, 4, 4, TGF, W, 0, TAU]
        - 2 * ff[:, :, 0, 0, TGF, W, 0, TAU]
    )

    # Nucleon 2pt
    # Original shape:
    # [cfg, tsrc, xsrc, ysrc, zsrc, pf, tsep]

    B = p["pt2_forward"][
        :, :, :, :, :, PF, TSEP
    ]

    # Average spatial sources
    # Shape: [cfg, tsrc]
    B = B.mean(axis=(2, 3, 4))


assert B.shape == G.shape

Ncfg, Nsrc = B.shape

print(f"Ncfg = {Ncfg}")
print(f"Nsrc = {Nsrc}")
print(f"B shape = {B.shape}")
print(f"G shape = {G.shape}")


# ============================================================
# 4. Original vacuum-subtracted 3pt (Method 1)
# ============================================================

# Average over tsrc within each configuration

BG_cfg = np.mean(B * G, axis=1)
B_cfg = np.mean(B, axis=1)
G_cfg = np.mean(G, axis=1)

# Original full-sample estimator
Y_full = (
    np.mean(BG_cfg)
    - np.mean(B_cfg) * np.mean(G_cfg)
)

# Jackknife
Y_jk_complex = (
    jackknife(BG_cfg)
    - jackknife(B_cfg) * jackknife(G_cfg)
)

# Select real or imaginary part
if PART == "real":
    Y_jk = Y_jk_complex.real
else:
    Y_jk = Y_jk_complex.imag

error_original = jk_error(Y_jk)


# ============================================================
# 5. Your proposed auxiliary variable
#
# Z_i = FF_i - <FF>
#
# Here <FF> is averaged over configurations
# separately for each tsrc.
# ============================================================

# [1, tsrc]
G_mean_cfg = np.mean(G, axis=0, keepdims=True)

# [cfg, tsrc]
Z = G - G_mean_cfg

# Average over tsrc
# [cfg]
Z_cfg = np.mean(Z, axis=1)

# Jackknife, keeping the full-sample FF mean fixed
Z_jk_complex = jackknife(Z_cfg)

if PART == "real":
    Z_jk = Z_jk_complex.real
else:
    Z_jk = Z_jk_complex.imag

print("\nAuxiliary variable checks:")
print("Full sample mean Z =", np.mean(Z_cfg))
print("JK error of Z =", jk_error(Z_jk))


# ============================================================
# 6. Calculate optimal alpha
# ============================================================

Y_centered = Y_jk - np.mean(Y_jk)
Z_centered = Z_jk - np.mean(Z_jk)

cov_YZ = np.sum(Y_centered * Z_centered)
var_Z = np.sum(Z_centered**2)

if var_Z == 0:
    raise ValueError("Auxiliary variable has zero variance.")

alpha_opt = cov_YZ / var_Z

rho = cov_YZ / np.sqrt(
    np.sum(Y_centered**2) * var_Z
)

print("\nControl variate:")
print(f"alpha_opt = {alpha_opt:.8e}")
print(f"Correlation coefficient = {rho:.6f}")


# ============================================================
# 7. Your proposed improved JK estimator
# ============================================================

Y_jk_proposed = Y_jk - alpha_opt * Z_jk

error_proposed = jk_error(Y_jk_proposed)


# ============================================================
# 8. Correct JK: recompute <FF> in each JK subset
# ============================================================

Z_jk_correct = np.zeros(Ncfg, dtype=float)

for i in range(Ncfg):

    # Delete configuration i
    G_subset = np.delete(G, i, axis=0)

    # Recalculate <FF> using ONLY the remaining cfgs
    G_subset_mean = np.mean(
        G_subset, axis=0, keepdims=True
    )

    # Construct centered FF again
    Z_subset = G_subset - G_subset_mean

    # Average over cfg and tsrc
    z_value = np.mean(Z_subset)

    if PART == "real":
        Z_jk_correct[i] = z_value.real
    else:
        Z_jk_correct[i] = z_value.imag

Y_jk_correct = (
    Y_jk - alpha_opt * Z_jk_correct
)

error_correct = jk_error(Y_jk_correct)


# ============================================================
# 9. Compare results
# ============================================================

print("\n" + "=" * 60)
print("CONTROL VARIATE TEST RESULTS")
print("=" * 60)

print(f"Operator: O3 = TXTX + TYTY - 2 XYXY")
print(f"q = 0")
print(f"PF index = {PF}")
print(f"TGF index = {TGF}")
print(f"Wilson-line index = {W}")
print(f"TSEP = {TSEP}, TAU = {TAU}")
print(f"Component = {PART}")

print("\nOriginal full-sample 3pt:")
print(Y_full)

print("\nJackknife errors:")
print(f"Original       : {error_original:.8e}")
print(f"Your method    : {error_proposed:.8e}")
print(f"Recomputed JK  : {error_correct:.8e}")

print("\nError ratios:")
print(
    f"Your / Original = "
    f"{error_proposed/error_original:.6f}"
)
print(
    f"Correct / Original = "
    f"{error_correct/error_original:.6f}"
)

print("\nVariance reduction factors:")
print(
    f"Apparent = "
    f"{(error_original/error_proposed)**2:.4f}"
)
print(
    f"Recomputed = "
    f"{(error_original/error_correct)**2:.4f}"
)

print("\nConsistency checks:")
print(
    "Full-sample correction =",
    np.mean(Z_cfg)
)
print(
    "Max recomputed JK correction =",
    np.max(np.abs(Z_jk_correct))
)
print(
    "Max difference from original JK =",
    np.max(np.abs(Y_jk_correct - Y_jk))
)

print("=" * 60)

Ncfg = 799
Nsrc = 8
B shape = (799, 8)
G shape = (799, 8)

Auxiliary variable checks:
Full sample mean Z = (4.179663151530001e-16+9.379447558948201e-33j)
JK error of Z = 0.41678618844954857

Control variate:
alpha_opt = -4.14885874e-08
Correlation coefficient = -0.047031

CONTROL VARIATE TEST RESULTS
Operator: O3 = TXTX + TYTY - 2 XYXY
q = 0
PF index = 5
TGF index = 3
Wilson-line index = 1
TSEP = 2, TAU = 6
Component = real

Original full-sample 3pt:
(-3.759668435239589e-07-1.673057324193069e-07j)

Jackknife errors:
Original       : 3.67672936e-07
Your method    : 3.67266087e-07
Recomputed JK  : 3.67672936e-07

Error ratios:
Your / Original = 0.998893
Correct / Original = 1.000000

Variance reduction factors:
Apparent = 1.0022
Recomputed = 1.0000

Consistency checks:
Full-sample correction = (4.179663151530001e-16+9.379447558948201e-33j)
Max recomputed JK correction = 1.2109500258568123e-15
Max difference from original JK = 5.293955920339377e-23


In [7]:
#Smearing tuning run 4 (rho_z scan at rho_T 4.5 and boost 0.8, 10 operators, Delta, stream e): effective masses and fit-free overlap comparisons, pion and proton, one page per pz.
#For every particle and channel (G5, G5G4, G3G5 diagonal correlators, and the 3x3 gamma GEVP of each operator with solve_gevp of
#compare_3rhoz_pion.py: GEVP = reconstructed 2pt v0^dag C(t) v0, GEVP_lambda = principal correlator lambda_0(t, t0)), in plot_dir/<particle>/:
#  meff_<channel>.pdf         E_eff(t) = log C(t)/C(t+1) of every operator
#  dmeff_<channel>.pdf        E_eff(t) - E_eff,ref(t) on the same cfgs (correlated jackknife): below 0 = less excited-state contamination than the reference
#The overlap pages below are not made for GEVP_lambda: lambda_0 is normalized by C(t0) and carries no overlap factor.
#  ratio_<channel>.pdf        R(t) = [C(t)/C(t_norm)] / [C_ref(t)/C_ref(t_norm)]: tends to w(t_norm)/w_ref(t_norm), the ground-state fraction at t_norm
#                             relative to the reference (above 1 = more overlap); R(t_norm) = 1, it rises (falls) while E_eff(t) is below (above)
#                             E_eff,ref(t), and is flat once both operators are ground-state dominated
#  overlap_map_<channel>.pdf  R(T) against the noise (geometric mean over the last 4 usable t <= 10 of the E_eff error / median over the operators),
#                             all pz on one page
#and a table on stdout. C = (forward + s_a s_b backward) / 2 with s = +1, -1, +1 for G5, G5G4, G3G5 (tsep = 0: forward only), made hermitian in the
#gammas and multiplied by c_sign so that it is positive at tsep >= 1 (the G3G5 diagonal at tsep = 0 is a contact term of the other sign, t = 0 is
#not used); the t_src are averaged in each cfg, errors are jackknife over the cfgs.
import os
import warnings
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

run_dir = "/Users/mcp3270/Desktop/GLUON_ANALYSIS_MANUAL/2pt_production/delta/smear _tuning/run4_rhoz_scan"
data_dir = f"{run_dir}/data"
file_name = "2pt_coulomb_run4_rhoz_scan_10op_32src_200cfg_10ops_{particle}_tsrc_fb_tsep20_32src_ncfg30.h5"
plot_dir = f"{run_dir}/plots_ncfg10"

particles = ["pion", "proton"]
t0 = 2                                  # gamma GEVP: C(t) v = lambda C(t0) v
t_vec = 4                               # reconstructed 2pt: eigenvector at t_vec from the full average, kept fixed for the jackknife samples
t_norm = 3                              # ratio: C(t) / C(t_norm)
ref_row = [4.5, 2.25, 0.8]              # reference operator: the run-3 pick, op 0 of the scan
t_max_plot = 16
err_max = 0.1                           # meff points with a larger jackknife error are not drawn
diff_err_max = 0.1                      # the same for the E_eff difference and the ratio (0.05 for 50+ cfgs)
usable_err = 0.1                        # a t is usable when the median E_eff error over the operators is below this (0.05 for 50+ cfgs; 10 cfgs have
                                        # about 2.3 times the errors of the 55-cfg run 3)
m_disp = {"pion": 0.141, "proton": None}                   # am of the dispersion line (pion: compare_3rhoz_pion.py); None: no line
ylim_window = {"pion": (0.05, 0.3), "proton": (0.1, 0.5)}   # meff y range: center - lower, center + upper (center: dispersion, or the GEVP plateau)
Ls = 32
gamma_sign = {"G5": 1, "G5G4": -1, "G3G5": 1}             # time reflection C_ab(-t) = s_a s_b C_ab(t), as in the h5 attribute time_reflection
c_sign = {"pion": -1, "proton": 1}                        # the pion 2pt comes out negative in this contraction convention (all gammas, operators, pz, tsep >= 1)
noise_cut = 1.2                         # table: operators ranked by overlap among those with noise <= noise_cut (the run-1 cut)
map_noise_max = 3.0                     # overlap map: operators noisier than this (no signal at that pz) are listed, not drawn
warnings.filterwarnings("ignore", message="All-NaN slice encountered")     # t with no signal in any operator
warnings.filterwarnings("ignore", message="Mean of empty slice")


def jk(data, jk_axis=0):
    n = data.shape[jk_axis]
    data = np.moveaxis(data, jk_axis, 0)
    mean = np.mean(data, axis=0)
    err = np.sqrt((n - 1.0) / n * np.sum((data - mean) ** 2, axis=0))
    return mean, err

def meff(c):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.log(c[..., :-1] / c[..., 1:])

def as_str(x):
    return x.decode() if isinstance(x, bytes) else str(x)


def load(particle):
    #C (cfg, smear, g_snk, g_src, pf, tsep): forward-backward average, hermitian in (g_snk, g_src), times c_sign
    with h5py.File(f"{data_dir}/{file_name.format(particle=particle)}", "r") as f:
        assert as_str(f.attrs["dim_pt2"]) == "cfg, smear, gamma_sink, gamma_source, t_src, pf, tsep", f.attrs["dim_pt2"]
        pt2_forward = f["pt2_forward"][:].mean(axis=4)              # (cfg, smear, g_snk, g_src, pf, tsep), t_src averaged
        pt2_backward = f["pt2_backward"][:].mean(axis=4)
        pf_list = f["pf_list"][:]
        smear_list = f["smear_list"][:]
        gamma_list = [as_str(g) for g in f.attrs["gamma_list"]]
    s = np.array([gamma_sign[g] for g in gamma_list])
    C = 0.5 * (pt2_forward + s[None, None, :, None, None, None] * s[None, None, None, :, None, None] * pt2_backward)
    C[..., 0] = pt2_forward[..., 0]
    C = 0.5 * (C + np.conj(np.swapaxes(C, 2, 3)))                   # the G5-G3G5 and G5G4-G3G5 elements are imaginary
    C = c_sign[particle] * C
    return C, pf_list, smear_list, gamma_list


def solve_gevp(C_matrix):
    #from compare_3rhoz_pion.py. C_matrix (op_snk, op_src, pf, tsep), hermitian and positive; here op = gamma (n_op = 3) for one smearing.
    #GEVP at every t in t_list by Cholesky of C(t0): eigenvalues in decreasing order, eigenvectors v normalized to v^dag C(t0) v = 1.
    #Added: a pz whose C(t0) is not positive definite (no signal, e.g. the wide unboosted operators at large pz) is left NaN instead of stopping the run.
    eigen_val = np.full((n_pf, n_op, len(t_list)), np.nan)
    eigen_vec = np.full((n_pf, n_op, len(t_list), n_op), np.nan, "<c16")
    for ipf in range(n_pf):
        try:
            L = np.linalg.cholesky(C_matrix[:, :, ipf, t0])
        except np.linalg.LinAlgError:
            continue
        L_inv = np.linalg.inv(L)
        for it, t in enumerate(t_list):
            M = L_inv @ C_matrix[:, :, ipf, t] @ L_inv.conj().T
            w, u = np.linalg.eigh(M)
            v = L_inv.conj().T @ u
            order = np.argsort(w)[::-1]
            eigen_val[ipf, :, it] = w[order]
            eigen_vec[ipf, :, it] = v[:, order].T
    return eigen_val, eigen_vec


def gevp(C_mean, C_jk):
    #per smearing: reconstructed 2pt v0^dag C(t) v0 with the ground-state eigenvector v0 of the full average at t_vec (the same v0 for every jackknife
    #sample), and the principal correlator lambda_0(t), solved again on every jackknife sample; lambda_0 is NaN at t <= t0 (t_list starts at t0 + 1)
    rec_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    rec_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    lam_mean = np.full((n_smear, n_pf, n_tsep), np.nan)
    lam_jk = np.full((n_cfg, n_smear, n_pf, n_tsep), np.nan)
    it_vec = list(t_list).index(t_vec)
    for i in range(n_smear):
        eigen_val, eigen_vec = solve_gevp(C_mean[i])
        v0 = eigen_vec[:, 0, it_vec, :]                                     # (pf, gamma)
        rec_mean[i] = np.einsum("pa,abpT,pb->pT", np.conj(v0), C_mean[i], v0).real
        rec_jk[:, i] = np.einsum("pa,cabpT,pb->cpT", np.conj(v0), C_jk[:, i], v0).real
        lam_mean[i, :, t0 + 1:] = eigen_val[:, 0]
        lam_jk[:, i, :, t0 + 1:] = np.array([solve_gevp(C_jk[icfg, i])[0][:, 0] for icfg in range(n_cfg)])
        for ipf in np.where(~np.isfinite(eigen_val[:, 0, 0]))[0]:
            w0 = np.linalg.eigvalsh(C_mean[i, :, :, ipf, t0])
            print(f"  GEVP operator {i}, pz {pf_list[ipf][2]}: C(t0) not positive definite (eigenvalues / largest {np.round(w0 / w0.max(), 6)}), left NaN")
        n_fail = np.sum(~np.isfinite(lam_jk[:, i, :, t0 + 1]) & np.isfinite(eigen_val[None, :, 0, 0]))
        if n_fail:
            print(f"  GEVP operator {i}: C(t0) not positive definite on {n_fail} jackknife samples (sample, pz), their lambda_0 is NaN")
    deviation = np.nanmax(np.abs(rec_mean[:, :, t0] - 1))
    assert deviation < 1e-10, f"v0 is not normalized to v0^dag C(t0) v0 = 1 (max deviation {deviation:.1e})"
    return rec_mean, rec_jk, lam_mean, lam_jk


def styles(smear_list, i_ref):
    #rho_z scan: one colour map and marker per rho_T, the colour running with rho_z (viridis when there is a single rho_T); the reference in black
    cmaps = ["viridis", "Oranges", "Greens", "Purples", "Reds", "YlOrBr"]
    markers = ["o", "s", "^", "D", "v", "P"]
    rho_T_all = sorted({round(rT, 6) for rT, rz, fr in smear_list})
    out = []
    for i, (rT, rz, fr) in enumerate(smear_list):
        label = f"{i:2d}: [{rT:g}, {rz:g}, {fr:g}]"
        if i == i_ref:
            out.append(("black", "*", label + "  (ref)"))
            continue
        g = rho_T_all.index(round(rT, 6))
        members = sorted((j for j in range(len(smear_list)) if round(smear_list[j][0], 6) == rho_T_all[g] and j != i_ref), key=lambda j: smear_list[j][1])
        x = members.index(i) / max(len(members) - 1, 1)
        shade = 0.05 + 0.85 * x if cmaps[g % len(cmaps)] == "viridis" else 0.45 + 0.5 * x
        out.append((plt.get_cmap(cmaps[g % len(cmaps)])(shade), markers[g % len(markers)], label))
    return out


def page(pdf, t, y, e, style, title, ylabel, ylim, hline=None, hline_label=None, e_cut=None):
    fig, ax = plt.subplots(figsize=(28, 16), constrained_layout=True)
    n_op = y.shape[0]
    for i in range(n_op):
        color, marker, label = style[i]
        ok = np.isfinite(y[i]) & np.isfinite(e[i]) & (t >= 1) & (t <= t_max_plot)
        if e_cut is not None:
            ok &= e[i] < e_cut
        ax.errorbar(t[ok] + 0.02 * (i - (n_op - 1) / 2), y[i][ok], e[i][ok], fmt=marker + "-", color=color, ms=12 if marker == "*" else 9,
                    mec="black" if marker == "*" else color, mew=1.2, lw=2.0 if marker == "*" else 1.0, elinewidth=1.3, capsize=3, label=label)
    if hline is not None:
        ax.axhline(hline, color="gray", lw=1.5, ls="--", label=hline_label)
    ax.set_xlim(0.5, t_max_plot + 0.5)
    if np.all(np.isfinite(ylim)):
        ax.set_ylim(ylim)
    ax.set_xticks(np.arange(1, t_max_plot + 1))
    ax.tick_params(labelsize=18)
    ax.grid(alpha=0.3)
    ax.set_xlabel("t", fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_title(title, fontsize=24)
    ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=15, frameon=False)
    pdf.savefig(fig)
    plt.close(fig)


def range_of(vals, errs, e_cut, pad_frac=0.1):
    v = vals[np.isfinite(vals) & np.isfinite(errs) & (errs < e_cut)]
    if not v.size:
        return (np.nan, np.nan)
    lo, hi = np.percentile(v, [2, 98])
    pad = pad_frac * (hi - lo) + 1e-3
    return (lo - pad, hi + pad)


for particle in particles:
    C, pf_list, smear_list, gamma_list = load(particle)
    n_cfg, n_smear, _, _, n_pf, n_tsep = C.shape
    C_mean = C.mean(axis=0)
    C_jk = (C.sum(axis=0)[None] - C) / (n_cfg - 1)
    match = np.where(np.isclose(smear_list, ref_row).all(axis=1))[0]
    assert len(match) == 1, f"ref_row {ref_row} is not one of the operators in the file"
    i_ref = int(match[0])
    style = styles(smear_list, i_ref)
    out_dir = f"{plot_dir}/{particle}"
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n{particle}: {n_cfg} cfgs, {n_smear} operators, gammas {gamma_list}, pz {pf_list[:, 2].tolist()}, reference operator {i_ref} = {smear_list[i_ref]}")
    t_eff = np.arange(n_tsep - 1)
    t_all = np.arange(n_tsep)

    n_op = len(gamma_list)                                                  # GEVP basis of solve_gevp: the gammas of one smearing
    t_list = np.arange(t0 + 1, n_tsep)
    rec_mean, rec_jk, lam_mean, lam_jk = gevp(C_mean, C_jk)
    channels = {g: (C_mean[:, ig, ig].real, C_jk[:, :, ig, ig].real) for ig, g in enumerate(gamma_list)}
    channels["GEVP"] = (rec_mean, rec_jk)
    channels["GEVP_lambda"] = (lam_mean, lam_jk)
    meff_rec = meff(rec_mean)

    #center of the meff y range: dispersion relation, else the GEVP plateau at t = 6..10 (reference operator, all operators, G5-G5)
    def center(p):
        if m_disp[particle] is not None:
            p_lat = 2 * np.pi * pf_list[p][2] / Ls
            return 2 * np.arcsinh(np.sqrt(np.sinh(m_disp[particle] / 2) ** 2 + np.sin(p_lat / 2) ** 2))
        for m in [meff_rec[i_ref, p], meff_rec[:, p], meff(C_mean[:, 0, 0].real)[:, p]]:
            if np.isfinite(m[..., 6:11]).any():
                return np.nanmedian(m[..., 6:11])
        return np.nan

    for name, (c_mean, c_jk) in channels.items():
        what = {"GEVP": f"GEVP over {', '.join(gamma_list)} (t0 = {t0}, t_vec = {t_vec}), reconstructed 2pt",
                "GEVP_lambda": f"GEVP over {', '.join(gamma_list)} (t0 = {t0}), principal correlator lambda_0"}.get(name, f"{name}-{name}")
        y, yj = meff(c_mean), meff(c_jk)
        e = jk(yj)[1]
        dy = y - y[i_ref][None]
        de = jk(yj - yj[:, i_ref][:, None])[1]
        note0 = {p: ("   [pz = 0: no ground-state overlap" + (", P+ keeps only negative parity]" if particle == "proton" else "]"))
                 if name == "G3G5" and pf_list[p][2] == 0 else "" for p in range(n_pf)}
        with PdfPages(f"{out_dir}/meff_{name}.pdf") as pdf:
            for p in range(n_pf):
                c = center(p)
                lo, hi = ylim_window[particle]
                ylim = (c - lo, c + hi) if not note0[p] else range_of(y[:, p, 1:t_max_plot + 1], e[:, p, 1:t_max_plot + 1], err_max)
                page(pdf, t_eff, y[:, p], e[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff   {n_cfg} cfgs{note0[p]}",
                     "E_eff(t) = log lambda_0(t)/lambda_0(t+1)" if name == "GEVP_lambda" else "E_eff(t) = log C(t)/C(t+1)", ylim,
                     c if m_disp[particle] is not None else None, f"dispersion, am = {m_disp[particle]}", err_max)
        with PdfPages(f"{out_dir}/dmeff_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_eff, dy[:, p], de[:, p], style, f"{particle}  pz = {pf_list[p][2]}   {what}   E_eff - E_eff(op {i_ref}), same cfgs{note0[p]}",
                     "E_eff(t) - E_eff,ref(t)    (< 0: less excited-state contamination)", range_of(dy[:, p, 1:t_max_plot + 1], de[:, p, 1:t_max_plot + 1], diff_err_max),
                     0.0, None, diff_err_max)
        if name == "GEVP_lambda":
            continue

        with np.errstate(invalid="ignore", divide="ignore"):
            r = (c_mean / c_mean[:, :, t_norm:t_norm + 1]) / (c_mean[i_ref] / c_mean[i_ref, :, t_norm:t_norm + 1])[None]
            rj = (c_jk / c_jk[:, :, :, t_norm:t_norm + 1]) / (c_jk[:, i_ref] / c_jk[:, i_ref, :, t_norm:t_norm + 1])[:, None]
        re = jk(rj)[1]

        #per pz: usable t (median E_eff error below usable_err), noise = geometric mean of err/median over the last 4 usable t <= 10, overlap R at T
        noise = np.full((n_smear, n_pf), np.nan)
        R_T, R_T_err, T_used, t_last = np.full((n_smear, n_pf), np.nan), np.full((n_smear, n_pf), np.nan), {}, {}
        for p in range(n_pf):
            med = np.nanmedian(e[:, p], axis=0)
            usable = [t for t in range(t_norm, 11) if np.isfinite(med[t]) and med[t] <= usable_err]
            if not usable:
                continue
            win = usable[-4:]
            t_last[p] = usable[-1]
            with np.errstate(invalid="ignore", divide="ignore"):
                noise[:, p] = np.exp(np.nanmean(np.log(e[:, p, win] / med[win][None]), axis=1))
            T = min(7, usable[-1] + 1)
            if T >= t_norm + 2:                                             # R at t_norm + 1 is one step of E_eff: no overlap information
                T_used[p] = T
                R_T[:, p], R_T_err[:, p] = r[:, p, T], re[:, p, T]

        with PdfPages(f"{out_dir}/ratio_{name}.pdf") as pdf:
            for p in range(n_pf):
                page(pdf, t_all, r[:, p], re[:, p], style,
                     f"{particle}  pz = {pf_list[p][2]}   {what}   [C(t)/C({t_norm})] / same for op {i_ref}{note0[p]}",
                     f"ratio -> ground-state fraction at t = {t_norm} relative to op {i_ref}   (> 1: more overlap)",
                     range_of(r[:, p, 1:t_max_plot + 1], re[:, p, 1:t_max_plot + 1], diff_err_max), 1.0, None, diff_err_max)

        #overlap map: all pz on one page
        fig, axes = plt.subplots(2, 4, figsize=(30, 15), constrained_layout=True)
        for p in range(n_pf):
            ax = axes.flat[p]
            ax.grid(alpha=0.3)
            ax.axhline(1, color="gray", lw=1)
            ax.axvline(1, color="gray", lw=1, ls=":")
            ax.axvline(noise_cut, color="gray", lw=1, ls="--")
            ok = np.isfinite(R_T[:, p]) & np.isfinite(noise[:, p])
            x_lo = min(0.5, 0.8 * np.min(noise[ok, p], initial=0.5))
            for i in np.where(ok & (noise[:, p] <= map_noise_max))[0]:
                color, marker, label = style[i]
                x = noise[i, p]
                ax.errorbar(x, R_T[i, p], R_T_err[i, p], fmt=marker, color=color, ms=14 if marker == "*" else 10, mec="black", mew=0.8, elinewidth=1.2, capsize=0)
                ax.annotate(str(i), (x, R_T[i, p]), xytext=(5, 4), textcoords="offset points", fontsize=11)
            hidden = np.where(ok & (noise[:, p] > map_noise_max))[0]
            missing = np.where(~ok)[0] if p in T_used else []
            note = []
            if hidden.size:
                note.append(f"noise > {map_noise_max:g}, not drawn: " + ", ".join(f"{i} ({noise[i, p]:.0f})" for i in hidden))
            if len(missing):
                note.append("no R or noise (GEVP NaN): " + ", ".join(map(str, missing)))
            if note:
                ax.text(0.98, 0.02, "\n".join(note), transform=ax.transAxes, ha="right", va="bottom", fontsize=12,
                        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
            ax.set_xscale("log")
            ticks = [x_t for x_t in [0.2, 0.3, 0.5, 0.7, 1, noise_cut, 1.5, 2, 3] if x_t >= x_lo]
            ax.set_xticks(ticks)
            ax.set_xticklabels([f"{x_t:g}" for x_t in ticks])
            ax.minorticks_off()
            ax.set_xlim(x_lo, map_noise_max)                                # after set_xticks, which widens the view to its ticks
            ax.tick_params(labelsize=14)
            ax.set_title(f"pz = {pf_list[p][2]}   (R at T = {T_used.get(p, '-')})" + (note0[p] and "  no ground-state overlap"), fontsize=17)
            if p % 4 == 0:
                ax.set_ylabel(f"overlap: ground-state fraction at t = {t_norm} relative to op {i_ref}", fontsize=15)
            if p >= 3:
                ax.set_xlabel("noise: E_eff error / median over the operators", fontsize=15)
        leg = axes.flat[7]
        leg.axis("off")
        handles = [plt.Line2D([], [], marker=m, ls="", color=c, mec="black", mew=0.8, ms=14 if m == "*" else 10, label=l) for c, m, l in style]
        leg.legend(handles=handles, loc="center", fontsize=13, frameon=False)
        fig.suptitle(f"{particle}   {what}   {n_cfg} cfgs: top-left is best (more overlap, less noise)", fontsize=22)
        with PdfPages(f"{out_dir}/overlap_map_{name}.pdf") as pdf:
            pdf.savefig(fig)
        plt.close(fig)

        print(f"  {name}: op:R/noise, R = overlap relative to op {i_ref} at T, sorted by R among the operators with noise <= {noise_cut:g}")
        for p in range(n_pf):
            if p not in T_used:
                print(f"    pz {pf_list[p][2]}: " + (f"usable t only up to {t_last[p]}, T = {t_last[p] + 1} < t_norm + 2: no R" if p in t_last else "no usable t"))
                continue
            good = [i for i in np.argsort(-R_T[:, p]) if np.isfinite(R_T[i, p]) and noise[i, p] <= noise_cut]
            bad = [i for i in range(n_smear) if noise[i, p] > noise_cut]
            missing = [i for i in range(n_smear) if i not in good and i not in bad]
            print(f"    pz {pf_list[p][2]} (T = {T_used[p]}){note0[p]}: " + "  ".join(f"{i}:{R_T[i, p]:.3f}/{noise[i, p]:.2f}" for i in good[:7])
                  + f"   | noise > {noise_cut:g}: " + " ".join(map(str, bad)) + ("   | no R or noise (GEVP NaN): " + " ".join(map(str, missing)) if missing else ""))
    print(f"  wrote {out_dir}/")


pion: 30 cfgs, 10 operators, gammas ['G5', 'G5G4', 'G3G5'], pz [0, 1, 2, 3, 4, 5, 6], reference operator 0 = [4.5  2.25 0.8 ]
  GEVP operator 7: C(t0) not positive definite on 5 jackknife samples (sample, pz), their lambda_0 is NaN
  GEVP operator 8, pz 0: C(t0) not positive definite (eigenvalues / largest [-0.017068  0.13153   1.      ]), left NaN
  GEVP operator 9, pz 0: C(t0) not positive definite (eigenvalues / largest [-0.070615  0.153087  1.      ]), left NaN
  GEVP operator 9: C(t0) not positive definite on 1 jackknife samples (sample, pz), their lambda_0 is NaN
  G5: op:R/noise, R = overlap relative to op 0 at T, sorted by R among the operators with noise <= 1.2
    pz 0 (T = 7): 3:1.005/0.60  2:1.004/0.51  4:1.004/0.80  1:1.002/0.46  0:1.000/0.44   | noise > 1.2: 5 6 7 8 9
    pz 1 (T = 7): 5:1.024/1.18  4:1.020/0.82  3:1.016/0.63  2:1.011/0.53  1:1.006/0.47  0:1.000/0.44   | noise > 1.2: 6 7 8 9
    pz 2 (T = 7): 5:1.027/0.99  4:1.026/0.82  3:1.022/0.77  2:1.016/0.79  1:1.00